<a href="https://colab.research.google.com/github/AliGulzar-ml/dsa405-project/blob/main/DSA405_002_FA26_A5_agulzar2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5: Tidy Data and Reshaping

**DSA 405 · Week 5**

| | |
|---|---|
| **In class** | Friday, Sep 18 |
| **A5 due** | Thursday, Sep 24, 11:59 PM |
| **File** | `nc_schools_dirty.xlsx` |
| **Also this week** | **Bench Check 1** window opens (slots this week through Week 7) |
| **Time** | ~25 min in class, ~55 min at home |

## Overview

This week is built on Wickham's three rules for tidy data: every variable is a column,
every observation is a row, and every type of observational unit gets its own table.
They sound almost too simple to be useful, but the same three rules apply in R, SQL,
and Polars, and they explain something you have probably already noticed: a spreadsheet
formatted to look nice for human readers is hard for a computer to use.

The file we're working with is a school test-score workbook that violates many
standards: headers on row 4, three sheets, grade columns laid out sideways, footnote
markers inside the numbers, and one row that is not a school.

In [ ]:
# ---------------------------------------------------------------------------
# DSA 405 setup
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, requests, io

DATA = "https://raw.githubusercontent.com/jon-holt/DSA-405-Student/main/datasets/"
# DATA = "data/raw/"          # local users


def load(filename, kind="csv", **kw):
    """Read a class file whether DATA is a URL or a local folder."""
    path = DATA + filename
    if kind == "csv":
        return pd.read_csv(path, **kw)
    if kind == "excel":
        return pd.read_excel(path, **kw)
    if kind == "text":
        return requests.get(path, timeout=30).text if path.startswith("http") else open(path).read()
    if kind == "json":
        if path.startswith("http"):
            return requests.get(path, timeout=30).json()
        import json as _j
        return _j.load(open(path))
    raise ValueError(kind)


pd.set_option("display.width", 160)
print("pandas", pd.__version__)

pandas 2.2.3


---
# Part 1: Explore (in class)

## Task 1.1: Load it right, then look at the layout

We've seen this file's header problem before, in Week 2. Load it with the correct
header row, and this time look at the *shape* of the table rather than the numbers:

In [ ]:
schools = load("nc_schools_dirty.xlsx", "excel", header=3)

print(schools.shape)
print(list(schools.columns))
schools.head(3)

(97, 9)
['district', 'school', 'enrollment', 'grade_3_reading', 'grade_4_reading', 'grade_5_reading', 'grade_3_math', 'grade_4_math', 'grade_5_math']


,district,school,enrollment,grade_3_reading,grade_4_reading,grade_5_reading,grade_3_math,grade_4_math,grade_5_math
0,Harnett County,Reedy Creek Magnet Elementary,374,61.9*,53.5,56.1,57.2,59.6,54.1
1,Orange County,Dillard Middle School,274,64.3,58.4,53.7,59.8,47.7,53
2,Orange County,Dillard Magnet Elementary,119,<5,51,<5,<5,<5,<5


The layout violates the first rule. `grade_3_reading`, `grade_4_reading`,
`grade_5_reading`, and the rest of the measure columns each encode **two variables**,
grade level and subject, in the header row. A variable (grade) is spread across
columns, so an easy-sounding question like "average reading score by grade" makes you
need to read three columns instead of grouping just one.

Now check the sheets:

In [ ]:
all_sheets = load("nc_schools_dirty.xlsx", "excel", header=3, sheet_name=None)
for name, df in all_sheets.items():
    print(f"{name}: {df.shape}, columns: {list(df.columns)[:4]} ...")

SY2023-24: (97, 9), columns: ['district', 'school', 'enrollment', 'grade_3_reading'] ...
SY2024-25: (97, 9), columns: ['LEA', 'school_name', 'enrollment', 'g3_read'] ...
Notes: (9, 1), columns: ['  <5      Fewer than five students tested. Value suppressed for privacy.'] ...


The second sheet contains the same data for a second year under **different column
names** (`LEA`, `school_name`, `g3_read`), so two sheets that should be stacked into
one table cannot be combined until the column names are made to match. Expect to see this same problem in your own project data; it is very common when data comes from more than one system.
Today we work with the
2023–24 sheet.

## Task 1.2: Melt, wide to long

`melt` reshapes a table from wide to long. Here it turns the six grade/subject columns
into two: a *measure* name and a *value*.

In [ ]:
MEASURES = ["grade_3_reading", "grade_4_reading", "grade_5_reading",
            "grade_3_math", "grade_4_math", "grade_5_math"]

long = schools.melt(id_vars=["district", "school", "enrollment"],
                    value_vars=MEASURES,
                    var_name="measure", value_name="pct_proficient")

print(f"{len(schools)} rows x {len(MEASURES)} measures = {len(long)} rows")
assert len(long) == len(schools) * len(MEASURES)
long.head()

97 rows x 6 measures = 582 rows


,district,school,enrollment,measure,pct_proficient
0,Harnett County,Reedy Creek Magnet Elementary,374,grade_3_reading,61.9*
1,Orange County,Dillard Middle School,274,grade_3_reading,64.3
2,Orange County,Dillard Magnet Elementary,119,grade_3_reading,<5
3,Durham Public,Cary Magnet Elementary,832,grade_3_reading,48.9
4,Chapel Hill-Carrboro,Apex Friendship Elementary,578,grade_3_reading,66.5


Before trusting the reshape, verify the arithmetic: 97 × 6 = 582, and the assert
checks it. (An **assert** is a line of code that states something that must be true;
Python stops with an error message if it is not.) Reshaping only moves data around; it never adds or removes values. So the
row counts must multiply out exactly. If they do not, some rows were lost or duplicated
during the reshape, and you should find out which before going any further.

Questions that used to require reading three columns can now be answered with one
`groupby`, the pandas method that computes a summary (like a mean or a count) for
each group of rows:

In [ ]:
long[["grade", "subject"]] = long.measure.str.extract(r"grade_(\d)_(\w+)") ## creating 2 new columns for grade level and subject
scores = pd.to_numeric(long.pct_proficient, errors="coerce")

print(long.assign(v=scores).groupby("grade").v.mean().round(1))

grade
3    111.2
4    110.7
5    106.6
Name: v, dtype: float64


(The **regex**, a pattern that describes what a piece of text looks like, is the same
pattern from last week. `errors="coerce"` converted some cells to `NaN`, the value
pandas uses to mean "missing"; those cells are the subject of A5.)

---
## Checkpoint: submit before leaving class

1. Which of the 3 tidy rules does the wide layout violate? What specific question is difficult to answer because of the non-tidy layout?
2. What is the melt arithmetic for this sheet (rows before × number of measures = rows
   after), and did the assert pass?
3. One row of this sheet does not look like an observation. Name that row.




*Answers here.*


The wide layout violates the first tidy rule: every variable is a column. Specifically, variables like grade level and subject are encoded within column headers (e.g., grade_3_reading). A question that is difficult to answer because of this non-tidy layout is "average reading score by grade," as it requires reading multiple columns rather than grouping a single 'grade' column.

The melt arithmetic for this sheet is 97 rows (before) × 6 measures = 582 rows (after). Yes, the assert passed, confirming that the row counts multiplied out exactly.

The row that does not look like an observation is the TOTAL row, which represents an aggregate of all schools rather than a single school observation. This row has district as 'TOTAL' and school as 'ALL SCHOOLS'.



---
# Part 2: A5 (Tidy & Reshape)

Four tasks. They are the take-home half of the week.

## Task 2.1: The row that is not an observation

Rule two says every row is an observation. One row of this sheet is an aggregate (a
total computed from the other rows), not a school. If it stays in the table, every
statistic you compute from the table will be wrong.

1. Find it. The profiling methods from Week 2 will find it quickly, and so will sorting
   the table.
2. Show what the extra row does to a statistic: report the **maximum grade-3 reading
   score** twice, once with the row included and once with it removed. One of the two
   numbers is larger than 100, so it cannot be a percentage. That is strong evidence
   you found the right row.
3. Remove the row, report how many actual schools remain, and record the removal in the
   cleaning log with a count and a reason.

In [ ]:
# Identify the aggregate row and compare the maximum before and after removal.
MEASURES = ["grade_3_reading", "grade_4_reading", "grade_5_reading",
            "grade_3_math", "grade_4_math", "grade_5_math"]
is_total = (schools["district"].astype("string").str.strip().eq("TOTAL")
            & schools["school"].astype("string").str.strip().eq("ALL SCHOOLS"))
assert is_total.sum() == 1, "Expected exactly one TOTAL / ALL SCHOOLS row"

schools_clean = schools.loc[~is_total].copy()
reading_all = pd.to_numeric(
    schools["grade_3_reading"].astype("string").str.replace("*", "", regex=False),
    errors="coerce"
)
reading_schools = pd.to_numeric(
    schools_clean["grade_3_reading"].astype("string").str.replace("*", "", regex=False),
    errors="coerce"
)
print("Aggregate row:")
print(schools.loc[is_total, ["district", "school", "enrollment", "grade_3_reading"]].to_string(index=False))
print(f"Maximum grade-3 reading with TOTAL: {reading_all.max():,.1f}%")
print(f"Maximum grade-3 reading without TOTAL: {reading_schools.max():.1f}%")
print(f"Actual schools remaining: {len(schools_clean)}")
cleaning_log = pd.DataFrame([{
    "action": "Remove aggregate row", "rows_removed": int(is_total.sum()),
    "reason": "TOTAL / ALL SCHOOLS is a column sum, not an individual school."
}])
print("\nCleaning log:")
print(cleaning_log.to_string(index=False))

Aggregate row:
district      school  enrollment grade_3_reading
   TOTAL ALL SCHOOLS       45839          4119.8
Maximum grade-3 reading with TOTAL: 4,119.8%
Maximum grade-3 reading without TOTAL: 80.7%
Actual schools remaining: 96

Cleaning log:
              action  rows_removed                                                         reason
Remove aggregate row             1 TOTAL / ALL SCHOOLS is a column sum, not an individual school.


### Task 2.1 — Result

The `TOTAL / ALL SCHOOLS` row is an aggregate, so its **4,119.8%** grade-3 reading entry is a sum rather than a possible school percentage. Removing that one row leaves **96 actual schools**, and the highest school grade-3 reading score is **80.7%**. The cleaning log above records the one-row removal and its reason.

## Task 2.2: Four markers, four meanings

The score cells contain more than numbers. Collect every **non-numeric marker** in the
six measure columns; a marker here is a symbol or short code written in a cell instead
of a number. The `Notes` sheet explains what each marker means. There are four:

For each marker, report how many cells contain it, what it means, and what value it
should become in a numeric column. Then a question to think carefully about: **one of
the four markers is a different kind of marker from the other three.** Which one, and
why? (Consider whether a real value exists for the cell and the marker only hides it.) The distinction matters
because the correct numeric replacement depends on it.

In [ ]:
# Count markers in school score cells, excluding the TOTAL row.
raw_scores = schools_clean[MEASURES].astype("string").apply(lambda col: col.str.strip())
marker_counts = pd.Series({
    "<5": int(raw_scores.eq("<5").sum().sum()),
    "*": int(raw_scores.apply(lambda col: col.str.endswith("*", na=False)).sum().sum()),
    "†": int(raw_scores.eq("†").sum().sum()),
    "blank": int(raw_scores.isna().sum().sum() + raw_scores.eq("").sum().sum()),
}, name="cells")
print("Markers across the six measures (96 schools):")
print(marker_counts.to_string())

# Make sure every non-numeric score has been explained by one of these markers.
without_revision_flag = raw_scores.replace(r"\*$", "", regex=True)
recognized_missing = raw_scores.isna() | raw_scores.eq("") | raw_scores.isin(["<5", "†"])
unknown = without_revision_flag.where(~recognized_missing).stack().loc[
    lambda values: pd.to_numeric(values, errors="coerce").isna()
]
assert unknown.empty, f"Unexplained score values: {unknown.unique().tolist()}"

Markers across the six measures (96 schools):
<5       58
*        17
†         9
blank    16


### Task 2.2 — Marker interpretation

| Marker | Cells | Meaning from `Notes` | Numeric treatment |
|---|---:|---|---|
| `<5` | 58 | Fewer than five students tested; the score is suppressed for privacy. | Missing (`NaN`); never substitute 5 or 0. |
| `*` | 17 | The displayed score was revised after initial publication. | Remove `*` and keep the displayed numeric score (for example, `61.9*` → `61.9`). |
| `†` | 9 | The school did not report the data. | Missing (`NaN`). |
| Blank | 16 | The school does not offer that grade level. | Missing (`NaN`). |

`<5` is the distinct **hidden-value** case: a real score exists but cannot be shown. A dagger means no score was reported, while a blank means the grade is not offered. An asterisk is attached to a visible revised score, so that number stays in the analysis. These reasons for missing or revised data should remain distinguishable in the raw data even though the numeric analysis column uses `NaN` for the first, third, and fourth cases.

## Task 2.3: The question tidy makes easy

Using the tidy long table (TOTAL row removed, markers handled, values numeric): compute
the mean grade-3 reading proficiency **by district**, sorted. Report the highest and
lowest districts with their values.

Then reshape once more: use `pivot_table` to turn the district means back to wide,
districts as rows, measures as columns, and show the result. The two layouts have
different uses: the long table is the one you compute on, and the wide table is the one
a person reads.

In [ ]:
# Each observation in the long table is one school × one measure.
tidy = schools_clean.melt(
    id_vars=["district", "school", "enrollment"],
    value_vars=MEASURES,
    var_name="measure", value_name="raw_score"
)
tidy[["grade", "subject"]] = tidy["measure"].str.extract(r"grade_(\d)_(\w+)")
raw = tidy["raw_score"].astype("string").str.strip()
tidy["suppressed"] = raw.eq("<5").fillna(False)
tidy["pct_proficient"] = pd.to_numeric(
    raw.replace({"<5": pd.NA, "†": pd.NA, "": pd.NA})
       .str.replace("*", "", regex=False),
    errors="coerce"
)
assert len(tidy) == len(schools_clean) * len(MEASURES) == 576

grade3_reading = (
    tidy.loc[tidy["measure"].eq("grade_3_reading")]
        .groupby("district")["pct_proficient"].mean().sort_values()
)
print("Grade-3 reading mean by district, lowest to highest (%):")
print(grade3_reading.round(3).to_string())
print(f"\nLowest: {grade3_reading.index[0]} ({grade3_reading.iloc[0]:.3f}%)")
print(f"Highest: {grade3_reading.index[-1]} ({grade3_reading.iloc[-1]:.3f}%)")

wide_district_means = tidy.pivot_table(
    index="district", columns="measure", values="pct_proficient", aggfunc="mean"
).reindex(columns=MEASURES)
wide_district_means.columns.name = None
print("\nMean proficiency (%) by district and measure:")
print(wide_district_means.round(2).to_string())

Grade-3 reading mean by district, lowest to highest (%):
district
Harnett County          51.275
Franklin County         51.475
Durham Public           54.639
Chatham County          56.583
Johnston County           57.2
Orange County           57.354
Wake County             60.325
Chapel Hill-Carrboro     72.45

Lowest: Harnett County (51.275%)
Highest: Chapel Hill-Carrboro (72.450%)

Mean proficiency (%) by district and measure:
                      grade_3_reading  grade_4_reading  grade_5_reading  grade_3_math  grade_4_math  grade_5_math
district                                                                                                         
Chapel Hill-Carrboro            72.45            70.78            71.24         66.54         68.29         67.06
Chatham County                  56.58             58.7            51.04          56.7         52.26         53.16
Durham Public                   54.64            54.39            52.73         52.93         52.06         4

## Task 2.4: Who disappears when suppressed cells are dropped

The `<5` marker means *fewer than five students tested; value suppressed for privacy*
(the real value was removed so that no student can be identified). A common shortcut is
to drop every school that has any suppressed cell. This task measures what that
shortcut removes from the data.

1. Split the schools into two groups: schools with at least one suppressed cell, and
   schools with none. Report the **mean enrollment** of each group.
2. Compute mean proficiency across all reported cells, and again using only the
   never-suppressed schools. Report both.
3. Then write a paragraph: which schools disappear from the data (their size and
   location), which direction the average moves, and why the schools that disappear
   matter for a policy question about school performance. Propose one concrete
   reporting practice that keeps the suppressed schools visible. Concrete means someone
   else could follow it next month without asking you what you meant.

In [ ]:
# A school is suppressed if ANY of its six raw score cells is <5.
school_suppressed = raw_scores.eq("<5").any(axis=1)
assert school_suppressed.sum() + (~school_suppressed).sum() == len(schools_clean)
print(f"Schools with ≥1 suppressed cell: {school_suppressed.sum()}")
print(f"Schools without suppressed cells: {(~school_suppressed).sum()}")
print(f"Mean enrollment, suppressed group: "
      f"{schools_clean.loc[school_suppressed, 'enrollment'].mean():.2f}")
print(f"Mean enrollment, never-suppressed group: "
      f"{schools_clean.loc[~school_suppressed, 'enrollment'].mean():.2f}")

# Calculate cell-weighted means of the reported numeric scores (NaNs are ignored).
all_reported = tidy["pct_proficient"].mean()
# Original row indices survive melt as a repeating block for each measure.
tidy["school_suppressed"] = pd.concat([school_suppressed] * len(MEASURES), ignore_index=True).to_numpy()
never_suppressed_mean = tidy.loc[~tidy["school_suppressed"], "pct_proficient"].mean()
print(f"\nAll reported school-measure cells: {all_reported:.2f}% "
      f"({tidy['pct_proficient'].count()} cells)")
print(f"Only never-suppressed schools: {never_suppressed_mean:.2f}% "
      f"({tidy.loc[~tidy['school_suppressed'], 'pct_proficient'].count()} cells)")
print(f"Difference after dropping schools: {never_suppressed_mean - all_reported:+.2f} percentage points")
print("\nNumber of schools with any suppression by district:")
print(schools_clean.loc[school_suppressed].groupby("district").size().sort_values(ascending=False).to_string())

Schools with ≥1 suppressed cell: 30
Schools without suppressed cells: 66
Mean enrollment, suppressed group: 242.23
Mean enrollment, never-suppressed group: 584.42

All reported school-measure cells: 55.39% (493 cells)
Only never-suppressed schools: 56.37% (375 cells)
Difference after dropping schools: +0.98 percentage points

Number of schools with any suppression by district:
district
Chatham County          8
Durham Public           5
Wake County             5
Franklin County         4
Orange County           4
Johnston County         2
Chapel Hill-Carrboro    1
Harnett County          1


### Task 2.4 — Interpretation

Dropping every school with even one `<5` cell removes **30 of 96 schools**, including many smaller schools: their mean enrollment is **242.23**, versus **584.42** for the 66 schools with no suppressed cells. The removed schools are spread across all eight districts, with the largest counts in **Chatham County (8)**, **Durham Public (5)**, and **Wake County (5)**. Across reported school-measure cells, the mean rises from **55.39%** to **56.37%** when those 30 schools are excluded, an increase of **0.98 percentage points**. This comparison does not prove why their scores differ, but it shows that deleting entire schools changes the group represented by the average. A policy maker could overlook smaller schools and the districts where many scores are suppressed. A reproducible reporting practice is to keep all 96 schools in the school-level table, show `<5` scores as missing with a separate suppression flag, and publish each district's count of included schools and suppressed cells beside its mean of the reported numeric cells. These are unweighted averages of available cells, not student-weighted district proficiency rates.

---
## AI use note

I used ChatGPT to help write the pandas code, check the marker counts and calculations against the workbook, and format the explanations in Part 2. I reviewed the interpretations and retained the original assignment structure.

---
## Submitting

1. **Runtime > Restart runtime**, then **Run all**.
2. `File > Download > Download .ipynb`
3. Rename to `DSA405_002_FA26_A5_[yourUnityID].ipynb`
4. Upload to the **A5** space on Moodle.

The **Checkpoint** section goes separately to **Week 5 In-Class Activity** before the
end of class on Friday. A5 is due **Thursday, Sep 24, 11:59 PM**.